<a href="https://colab.research.google.com/github/TALHANASEER-GIS/geoai-solar-potential/blob/main/solar_panel_geoai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# Updated STEP 1: Install all packages including the missing map server
!pip install segment-geospatial leafmap geopandas rasterio localtileserver


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 593.1/593.1 kB 47.7 MB/s eta 0:00:00


In [16]:
# STEP 2: Create a local sample image directly (No download needed!)
import numpy as np
import rasterio
from rasterio.transform import from_origin
import leafmap

image_path = "satellite_scene.tif"

# Create a synthetic 3-band (RGB) image representing a house rooftop and yard
# Size: 400x400 pixels
arr = np.zeros((3, 400, 400), dtype=np.uint8)
arr[0, :, :] = 100  # Green background yard (Red channel)
arr[1, :, :] = 150  # Green background yard (Green channel)
arr[2, :, :] = 50   # Green background yard (Blue channel)

# Draw a red/orange rooftop structure in the middle
arr[0, 100:300, 100:300] = 220
arr[1, 100:300, 100:300] = 110
arr[2, 100:300, 100:300] = 50

# Draw dark solar panels on top of the roof
arr[0, 130:200, 130:270] = 20
arr[1, 130:200, 130:270] = 30
arr[2, 130:200, 130:270] = 60

# Define geographical positioning (centered near Salzburg coordinates)
transform = from_origin(13.04, 47.80, 0.00001, 0.00001)

# Save the generated array as a real GeoTIFF map file
with rasterio.open(
    image_path, 'w', driver='GTiff',
    height=arr.shape[1], width=arr.shape[2],
    count=3, dtype=arr.dtype,
    crs='EPSG:4326', transform=transform
) as dst:
    dst.write(arr)

print("Successfully created local satellite_scene.tif without any downloads!")

# Show it on your map canvas
m = leafmap.Map(center=[47.80, 13.04], zoom=15)
m.add_raster(image_path, layer_name="Synthetic Satellite Scene")
m


Successfully created local satellite_scene.tif without any downloads!


Map(center=[47.798, 13.042], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom…

In [18]:
# STEP 3: Run the GeoAI Segmentation and Calculate Energy Potential
import os
import geopandas as gpd
from samgeo import SamGeo
from samgeo.common import raster_to_vector

print("Initializing Segment Anything Model (SAM)... Please wait a moment.")

# 1. Initialize the AI Model engine
sam = SamGeo(
    model_type="vit_b",  # Using vit_b for faster cloud execution
    checkpoint="sam_vit_b_01ec2a.pth",
    sam_kwargs=None,
)

# 2. Run the AI on your local image to segment the roof/solar structures
output_mask = "solar_masks.tif"
sam.generate(image_path, output=output_mask, erosion_kernel=(3, 3))

# 3. Convert the AI pixel arrays (Raster) into GIS vector shapes (Polygons)
vector_output = "final_solar_potential.geojson"
raster_to_vector(output_mask, vector_output)

# 4. Read the data layers using GeoPandas
gdf = gpd.read_file(vector_output)

# Project the map shapes into an accurate European metric format (EPSG:31287 - Austria Lambert)
gdf = gdf.to_crs(epsg=31287)
gdf['area_sqm'] = gdf.geometry.area

# Filter out tiny pixel noise or artifacts smaller than 3 square meters
gdf_clean = gdf[gdf['area_sqm'] > 3.0]

# Calculate energy generation potential (Area * 1200 kWh/m² * 20% panel efficiency / 1000 for Megawatt Hours)
gdf_clean['annual_mwh'] = (gdf_clean['area_sqm'] * 1200 * 0.20) / 1000

print("\n--- GEOAI METRICS DETECTED ---")
print(f"Total Structural Footprints Isolated: {len(gdf_clean)}")
for idx, row in gdf_clean.iterrows():
    print(f"Structure #{idx+1}: Area = {row['area_sqm']:.2f} m² | Estimated Capacity = {row['annual_mwh']:.2f} MWh/year")

# Save your final portfolio dataset
gdf_clean.to_file("final_solar_potential.geojson", driver="GeoJSON")
print("\nSuccess! Your portfolio spatial file 'final_solar_potential.geojson' is ready.")


Downloading...
From: https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
To: /root/.cache/torch/hub/checkpoints/sam_vit_b_01ec64.pth


Initializing Segment Anything Model (SAM)... Please wait a moment.
Model checkpoint for vit_b not found.


100%|██████████| 375M/375M [00:07<00:00, 52.9MB/s]



--- GEOAI METRICS DETECTED ---
Total Structural Footprints Isolated: 3
Structure #1: Area = 8156.90 m² | Estimated Capacity = 1957.66 MWh/year
Structure #2: Area = 25136.02 m² | Estimated Capacity = 6032.64 MWh/year
Structure #3: Area = 99101.28 m² | Estimated Capacity = 23784.31 MWh/year

Success! Your portfolio spatial file 'final_solar_potential.geojson' is ready.
